<a href="https://colab.research.google.com/github/njpinton/CMSC178IP/blob/main/06%20-%20Image%20Restoration%20and%20Geometric%20Processing/notebooks/image_restoration_workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔧 Image Restoration and Geometric Processing Workshop

**CMSC 178IP - Digital Image Processing**  
**Author:** Noel Jeffrey Pinton  
**University of the Philippines - Cebu**

---

## 🎯 Learning Objectives

By the end of this workshop, you will be able to:

1. **Understand** the fundamental degradation model in image processing
2. **Apply** various spatial and frequency domain restoration techniques
3. **Implement** noise reduction and deblurring algorithms
4. **Perform** geometric transformations and image registration
5. **Evaluate** restoration quality using appropriate metrics
6. **Solve** real-world restoration problems

---

## 📋 Workshop Overview

**Total Duration:** 45-60 minutes

1. **Setup & Imports** (5 min)
2. **Part 1:** Understanding Image Degradation (10 min)
3. **Part 2:** Spatial Domain Restoration (10 min)
4. **Part 3:** Frequency Domain Restoration (10 min)
5. **Part 4:** Geometric Processing (8 min)
6. **Student Activity:** Practical Restoration Challenge (15 min)
7. **Solutions & Discussion** (7 min)

---

## 🛠️ Setup & Imports

Let's start by importing the necessary libraries and setting up our environment.

In [ ]:
# Essential imports for image restoration and geometric processing
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy import ndimage, signal
from skimage import data, restoration, transform, filters, util, measure
from skimage.metrics import structural_similarity as ssim
from skimage.restoration import inpaint
import warnings
warnings.filterwarnings('ignore')

# Set up matplotlib for nice visualizations
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Color scheme for consistent plots
PRIMARY_BLUE = '#144B8C'
SECONDARY_BLUE = '#4682B4'
ACCENT_ORANGE = '#E65F2D'

print("🎉 Setup complete! Ready for image restoration workshop.")

## 📸 Part 1: Understanding Image Degradation

Before we can restore images, we need to understand how they get degraded. The fundamental degradation model is:

$$g(x,y) = h(x,y) * f(x,y) + \eta(x,y)$$

Where:
- $f(x,y)$ = original image
- $h(x,y)$ = degradation function (PSF)
- $\eta(x,y)$ = additive noise
- $g(x,y)$ = degraded image
- $*$ = convolution operator

In [ ]:
# Load a test image - use astronaut for rich texture and detail
original_image = data.astronaut()
# Convert to grayscale for processing
original_image = cv2.cvtColor(original_image, cv2.COLOR_RGB2GRAY)

# Create different types of noise
gaussian_noisy = util.random_noise(original_image, mode='gaussian', var=0.01)
sp_noisy = util.random_noise(original_image, mode='s&p', amount=0.05)
speckle_noisy = util.random_noise(original_image, mode='speckle', var=0.1)

# Create motion blur
motion_kernel = np.zeros((15, 15))
motion_kernel[7, :] = 1/15  # Horizontal motion
blurred_image = signal.convolve2d(original_image, motion_kernel, mode='same', boundary='symm')

# Visualize degradations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Image Degradation Examples', fontsize=16, fontweight='bold')

axes[0, 0].imshow(original_image, cmap='gray')
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

axes[0, 1].imshow(gaussian_noisy, cmap='gray')
axes[0, 1].set_title('Gaussian Noise')
axes[0, 1].axis('off')

axes[0, 2].imshow(sp_noisy, cmap='gray')
axes[0, 2].set_title('Salt & Pepper Noise')
axes[0, 2].axis('off')

axes[1, 0].imshow(speckle_noisy, cmap='gray')
axes[1, 0].set_title('Speckle Noise')
axes[1, 0].axis('off')

axes[1, 1].imshow(blurred_image, cmap='gray')
axes[1, 1].set_title('Motion Blur')
axes[1, 1].axis('off')

# Combined degradation
combined_degraded = util.random_noise(blurred_image, mode='gaussian', var=0.005)
axes[1, 2].imshow(combined_degraded, cmap='gray')
axes[1, 2].set_title('Blur + Noise')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print("✅ Degradation models demonstrated!")
print(f"📊 Original image shape: {original_image.shape}")
print(f"📊 Image data type: {original_image.dtype}")
print(f"📊 Value range: [{original_image.min():.2f}, {original_image.max():.2f}]")

### 🔍 Analysis Questions:

1. **Which type of noise affects edges the most?**
2. **Why is the combination of blur and noise particularly challenging?**
3. **What information is lost due to motion blur vs. noise?**

## 🔧 Part 2: Spatial Domain Restoration

Spatial domain methods work directly on pixel values. Let's explore different filtering techniques for noise reduction.

In [ ]:
# Use the noisy images from Part 1
test_image = (gaussian_noisy * 255).astype(np.uint8)

# Apply different spatial filters
# 1. Mean filter
mean_filtered = ndimage.uniform_filter(test_image, size=3)

# 2. Gaussian filter
gaussian_filtered = ndimage.gaussian_filter(test_image, sigma=1.0)

# 3. Median filter
median_filtered = ndimage.median_filter(test_image, size=3)

# 4. Bilateral filter
bilateral_filtered = cv2.bilateralFilter(test_image, 9, 75, 75)

# Calculate quality metrics
def calculate_psnr(original, restored):
    mse = np.mean((original.astype(float) - restored.astype(float)) ** 2)
    if mse == 0:
        return float('inf')
    return 20 * np.log10(255.0 / np.sqrt(mse))

def calculate_ssim(original, restored):
    return ssim(original, restored, data_range=255)

# Evaluate results
original_uint8 = (original_image).astype(np.uint8)
filters_results = [
    ('Original', original_uint8),
    ('Noisy', test_image),
    ('Mean Filter', mean_filtered.astype(np.uint8)),
    ('Gaussian Filter', gaussian_filtered.astype(np.uint8)),
    ('Median Filter', median_filtered.astype(np.uint8)),
    ('Bilateral Filter', bilateral_filtered)
]

# Visualize results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Spatial Domain Restoration Results', fontsize=16, fontweight='bold')

for i, (name, img) in enumerate(filters_results):
    row, col = i // 3, i % 3
    axes[row, col].imshow(img, cmap='gray')
    
    if name != 'Original':
        psnr_val = calculate_psnr(original_uint8, img)
        ssim_val = calculate_ssim(original_uint8, img)
        axes[row, col].set_title(f'{name}\nPSNR: {psnr_val:.2f} dB, SSIM: {ssim_val:.3f}')
    else:
        axes[row, col].set_title(name)
    
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("✅ Spatial domain restoration complete!")
print("\n📊 Filter Performance Summary:")
for name, img in filters_results[2:]:  # Skip original and noisy
    psnr_val = calculate_psnr(original_uint8, img)
    ssim_val = calculate_ssim(original_uint8, img)
    print(f"  {name:15s}: PSNR = {psnr_val:6.2f} dB, SSIM = {ssim_val:.3f}")

In [ ]:
# Let's analyze filter behavior with different noise types
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle('Filter Performance on Different Noise Types', fontsize=16, fontweight='bold')

noise_types = [
    ('Gaussian', gaussian_noisy),
    ('Salt & Pepper', sp_noisy),
    ('Speckle', speckle_noisy)
]

for row, (noise_name, noisy_img) in enumerate(noise_types):
    noisy_uint8 = (noisy_img * 255).astype(np.uint8)
    
    # Apply filters
    mean_result = ndimage.uniform_filter(noisy_uint8, size=3)
    gaussian_result = ndimage.gaussian_filter(noisy_uint8, sigma=1.0)
    median_result = ndimage.median_filter(noisy_uint8, size=3)
    bilateral_result = cv2.bilateralFilter(noisy_uint8, 9, 75, 75)
    
    results = [noisy_uint8, mean_result, gaussian_result, median_result]
    titles = ['Noisy', 'Mean', 'Gaussian', 'Median']
    
    for col, (result, title) in enumerate(zip(results, titles)):
        axes[row, col].imshow(result, cmap='gray')
        
        if col == 0:
            axes[row, col].set_title(f'{noise_name} Noise')
        else:
            psnr_val = calculate_psnr(original_uint8, result)
            axes[row, col].set_title(f'{title}\nPSNR: {psnr_val:.1f} dB')
        
        axes[row, col].axis('off')
        
        # Add text label for noise type on the left
        if col == 0:
            axes[row, col].text(-0.1, 0.5, noise_name, rotation=90, 
                              transform=axes[row, col].transAxes, 
                              verticalalignment='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Filter comparison across noise types complete!")
print("\n🎯 Key Observations:")
print("  • Median filter excels at salt & pepper noise")
print("  • Gaussian filter works well for Gaussian noise")
print("  • Bilateral filter preserves edges while denoising")
print("  • Mean filter provides smoothing but blurs edges")

## 🌊 Part 3: Frequency Domain Restoration

Frequency domain methods use the Fourier transform to work in the frequency domain. This is particularly useful for deblurring operations.

In [ ]:
# Create a more realistic degradation scenario
# Start with the original image - use coins for detailed texture analysis
clean_image = data.coins().astype(float)

# Create a Gaussian blur kernel
def create_gaussian_kernel(size, sigma):
    kernel = np.zeros((size, size))
    center = size // 2
    for i in range(size):
        for j in range(size):
            x, y = i - center, j - center
            kernel[i, j] = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    return kernel / kernel.sum()

# Create degradation
blur_kernel = create_gaussian_kernel(15, 3)
blurred = signal.convolve2d(clean_image, blur_kernel, mode='same', boundary='symm')
noisy_blurred = blurred + np.random.normal(0, 5, blurred.shape)

# Frequency domain restoration
def wiener_filter(degraded_fft, psf_fft, noise_power=0.01):
    """Apply Wiener filter in frequency domain."""
    # Wiener filter formula
    psf_conj = np.conj(psf_fft)
    psf_mag_sq = np.abs(psf_fft) ** 2
    
    wiener = psf_conj / (psf_mag_sq + noise_power)
    restored_fft = degraded_fft * wiener
    
    return restored_fft

# Prepare for frequency domain processing
h, w = clean_image.shape

# Pad the PSF to image size
psf_padded = np.zeros_like(clean_image)
kh, kw = blur_kernel.shape
psf_padded[:kh, :kw] = blur_kernel

# FFT of images and PSF
clean_fft = np.fft.fft2(clean_image)
degraded_fft = np.fft.fft2(noisy_blurred)
psf_fft = np.fft.fft2(psf_padded)

# Apply Wiener filter
restored_fft = wiener_filter(degraded_fft, psf_fft, noise_power=0.01)
restored_wiener = np.real(np.fft.ifft2(restored_fft))

# Richardson-Lucy deconvolution (iterative method)
def richardson_lucy(image, psf, iterations=10):
    """Richardson-Lucy deconvolution algorithm."""
    # Initialize with the degraded image
    estimate = image.copy()
    psf_flipped = np.flip(psf)
    
    for i in range(iterations):
        # Forward model
        convolved = signal.convolve2d(estimate, psf, mode='same', boundary='symm')
        
        # Ratio
        ratio = image / (convolved + 1e-10)
        
        # Update estimate
        correction = signal.convolve2d(ratio, psf_flipped, mode='same', boundary='symm')
        estimate = estimate * correction
        
        # Ensure positivity
        estimate = np.maximum(estimate, 0)
    
    return estimate

# Apply Richardson-Lucy
restored_rl = richardson_lucy(np.maximum(noisy_blurred, 0), blur_kernel, iterations=20)

# Visualize results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Frequency Domain Restoration', fontsize=16, fontweight='bold')

images = [
    ('Original', clean_image),
    ('Blurred + Noise', noisy_blurred),
    ('Wiener Filter', restored_wiener),
    ('Richardson-Lucy', restored_rl),
    ('Gaussian Filter', ndimage.gaussian_filter(noisy_blurred, sigma=1.0)),
    ('Unsharp Mask', clean_image)  # Placeholder
]

# Apply unsharp masking
gaussian_blurred = ndimage.gaussian_filter(noisy_blurred, sigma=2)
unsharp_mask = noisy_blurred - gaussian_blurred
unsharp_result = noisy_blurred + 2 * unsharp_mask
images[5] = ('Unsharp Mask', unsharp_result)

for i, (name, img) in enumerate(images):
    row, col = i // 3, i % 3
    axes[row, col].imshow(img, cmap='gray', vmin=0, vmax=255)
    
    if name != 'Original':
        psnr_val = calculate_psnr(clean_image, img)
        axes[row, col].set_title(f'{name}\nPSNR: {psnr_val:.2f} dB')
    else:
        axes[row, col].set_title(name)
    
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("✅ Frequency domain restoration complete!")
print("\n📊 Restoration Performance:")
for name, img in images[2:]:  # Skip original and degraded
    psnr_val = calculate_psnr(clean_image, img)
    ssim_val = calculate_ssim(clean_image.astype(np.uint8), 
                             np.clip(img, 0, 255).astype(np.uint8))
    print(f"  {name:15s}: PSNR = {psnr_val:6.2f} dB, SSIM = {ssim_val:.3f}")

In [ ]:
# Let's visualize the frequency domain representations
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Frequency Domain Analysis', fontsize=16, fontweight='bold')

# Original image and its FFT
axes[0, 0].imshow(clean_image, cmap='gray')
axes[0, 0].set_title('Original Image')
axes[0, 0].axis('off')

clean_fft_shifted = np.fft.fftshift(clean_fft)
axes[0, 1].imshow(np.log(1 + np.abs(clean_fft_shifted)), cmap='viridis')
axes[0, 1].set_title('Original FFT (Log Scale)')
axes[0, 1].axis('off')

# Degraded image and its FFT
axes[0, 2].imshow(noisy_blurred, cmap='gray')
axes[0, 2].set_title('Degraded Image')
axes[0, 2].axis('off')

degraded_fft_shifted = np.fft.fftshift(degraded_fft)
axes[0, 3].imshow(np.log(1 + np.abs(degraded_fft_shifted)), cmap='viridis')
axes[0, 3].set_title('Degraded FFT (Log Scale)')
axes[0, 3].axis('off')

# PSF and its frequency response
axes[1, 0].imshow(blur_kernel, cmap='hot')
axes[1, 0].set_title('Blur Kernel (PSF)')
axes[1, 0].axis('off')

psf_fft_shifted = np.fft.fftshift(psf_fft)
axes[1, 1].imshow(np.abs(psf_fft_shifted), cmap='viridis')
axes[1, 1].set_title('PSF Frequency Response')
axes[1, 1].axis('off')

# Restored image and its FFT
axes[1, 2].imshow(restored_wiener, cmap='gray')
axes[1, 2].set_title('Wiener Restored')
axes[1, 2].axis('off')

restored_fft_shifted = np.fft.fftshift(restored_fft)
axes[1, 3].imshow(np.log(1 + np.abs(restored_fft_shifted)), cmap='viridis')
axes[1, 3].set_title('Restored FFT (Log Scale)')
axes[1, 3].axis('off')

plt.tight_layout()
plt.show()

print("✅ Frequency domain analysis complete!")
print("\n🎯 Key Insights:")
print("  • Blur attenuates high frequencies")
print("  • Wiener filter balances restoration and noise amplification")
print("  • Richardson-Lucy preserves positivity constraint")
print("  • Frequency domain reveals the nature of degradation")

## 🔄 Part 4: Geometric Processing

Geometric processing involves spatial transformations like rotation, scaling, and more complex warping operations.

In [ ]:
# Load a test image for geometric processing - use text for clear transformation visualization
test_img = data.text()

# Define various transformations
# 1. Translation
translation = transform.SimilarityTransform(translation=(50, 30))
translated = transform.warp(test_img, translation)

# 2. Rotation
rotated = transform.rotate(test_img, angle=30, resize=False)

# 3. Scaling
scaled = transform.rescale(test_img, 1.3, anti_aliasing=True)

# 4. Affine transformation (combination)
affine_matrix = np.array([[1.2, 0.3, 20],
                         [0.1, 1.1, 15],
                         [0, 0, 1]])
affine_transform = transform.AffineTransform(matrix=affine_matrix)
affine_warped = transform.warp(test_img, affine_transform)

# 5. Perspective transformation
corners = np.array([[0, 0], [0, 200], [200, 0], [200, 200]])
new_corners = np.array([[20, 10], [10, 180], [180, 20], [190, 190]])
perspective_transform = transform.ProjectiveTransform()
perspective_transform.estimate(corners, new_corners)
perspective_warped = transform.warp(test_img, perspective_transform, output_shape=(200, 200))

# Demonstrate interpolation methods using binary blobs for clear comparison
np.random.seed(42)  # For reproducible results
small_img = data.binary_blobs(length=50, blob_size_fraction=0.1, n_dim=2, volume_fraction=0.3)

# Different interpolation methods for upsampling
nearest = transform.resize(small_img, (200, 200), order=0, anti_aliasing=False)
bilinear = transform.resize(small_img, (200, 200), order=1, anti_aliasing=False)
bicubic = transform.resize(small_img, (200, 200), order=3, anti_aliasing=False)

# Visualize geometric transformations
fig, axes = plt.subplots(3, 3, figsize=(15, 15))
fig.suptitle('Geometric Transformations and Interpolation', fontsize=16, fontweight='bold')

transformations = [
    ('Original', test_img),
    ('Translation', translated),
    ('Rotation (30°)', rotated),
    ('Scaling (1.3×)', scaled),
    ('Affine Transform', affine_warped),
    ('Perspective', perspective_warped),
    ('Nearest Neighbor', nearest),
    ('Bilinear', bilinear),
    ('Bicubic', bicubic)
]

for i, (name, img) in enumerate(transformations):
    row, col = i // 3, i % 3
    axes[row, col].imshow(img, cmap='gray')
    axes[row, col].set_title(name)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("✅ Geometric processing demonstration complete!")
print("\n📊 Transformation Properties:")
print(f"  Original shape: {test_img.shape}")
print(f"  Scaled shape: {scaled.shape}")
print(f"  Small image shape: {small_img.shape}")
print("\n🎯 Key Concepts:")
print("  • Affine transformations preserve parallelism")
print("  • Perspective transformations can correct distortion")
print("  • Interpolation quality affects final image quality")
print("  • Anti-aliasing reduces artifacts in scaling")

In [ ]:
# Demonstrate image registration using retina image for medical registration example
# Create a reference image and a moved version
reference = data.retina()
# Convert to grayscale for registration
reference = cv2.cvtColor(reference, cv2.COLOR_RGB2GRAY)

# Create a transformed version (simulating misaligned images)
transform_params = transform.SimilarityTransform(rotation=np.radians(5), 
                                               translation=(15, -10),
                                               scale=0.95)
moving = transform.warp(reference, transform_params, preserve_range=True)

# Add some noise to make it more realistic
moving = moving + np.random.normal(0, 5, moving.shape)
moving = np.clip(moving, 0, 255)

# Simple correlation-based registration
def register_images(ref_img, moving_img):
    """Simple translation registration using normalized cross-correlation."""
    # Convert to frequency domain
    ref_fft = np.fft.fft2(ref_img)
    moving_fft = np.fft.fft2(moving_img)
    
    # Cross-correlation in frequency domain
    cross_corr = ref_fft * np.conj(moving_fft)
    correlation = np.abs(np.fft.ifft2(cross_corr))
    
    # Find peak
    peak_idx = np.unravel_index(np.argmax(correlation), correlation.shape)
    
    # Convert to displacement
    h, w = ref_img.shape
    y_shift = peak_idx[0] if peak_idx[0] < h//2 else peak_idx[0] - h
    x_shift = peak_idx[1] if peak_idx[1] < w//2 else peak_idx[1] - w
    
    return -y_shift, -x_shift

# Perform registration
y_shift, x_shift = register_images(reference, moving)

# Apply correction
correction_transform = transform.SimilarityTransform(translation=(x_shift, y_shift))
registered = transform.warp(moving, correction_transform, preserve_range=True)

# Calculate differences
diff_before = np.abs(reference.astype(float) - moving.astype(float))
diff_after = np.abs(reference.astype(float) - registered.astype(float))

# Visualize registration results
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Medical Image Registration Example', fontsize=16, fontweight='bold')

# Before registration
axes[0, 0].imshow(reference, cmap='gray')
axes[0, 0].set_title('Reference Image')
axes[0, 0].axis('off')

axes[0, 1].imshow(moving, cmap='gray')
axes[0, 1].set_title('Moving Image')
axes[0, 1].axis('off')

axes[0, 2].imshow(diff_before, cmap='hot')
axes[0, 2].set_title(f'Difference (Before)\nMean Error: {np.mean(diff_before):.2f}')
axes[0, 2].axis('off')

# After registration
axes[1, 0].imshow(reference, cmap='gray')
axes[1, 0].set_title('Reference Image')
axes[1, 0].axis('off')

axes[1, 1].imshow(registered, cmap='gray')
axes[1, 1].set_title('Registered Image')
axes[1, 1].axis('off')

axes[1, 2].imshow(diff_after, cmap='hot')
axes[1, 2].set_title(f'Difference (After)\nMean Error: {np.mean(diff_after):.2f}')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

print(f"✅ Medical image registration complete!")
print(f"\n📊 Registration Results:")
print(f"  Detected shift: ({x_shift:.1f}, {y_shift:.1f}) pixels")
print(f"  Error before registration: {np.mean(diff_before):.2f}")
print(f"  Error after registration: {np.mean(diff_after):.2f}")
print(f"  Improvement: {(np.mean(diff_before) - np.mean(diff_after))/np.mean(diff_before)*100:.1f}%")

## 🎯 Student Activity: Practical Restoration Challenge

**Duration: 15 minutes**

### 📋 Your Mission:

You've been given a severely degraded image that suffers from multiple problems:
1. **Motion blur** from camera shake
2. **Gaussian noise** from low light conditions  
3. **Geometric distortion** from perspective

Your goal is to restore this image to the best quality possible using the techniques you've learned.

### 🔧 Available Tools:
- Spatial domain filters (mean, gaussian, median, bilateral)
- Frequency domain restoration (Wiener filter, Richardson-Lucy)
- Geometric transformations
- Quality metrics (PSNR, SSIM)

### 📊 Success Criteria:
- Achieve PSNR > 25 dB
- Achieve SSIM > 0.8
- Visually pleasing result

### ⏰ Time Management:
- **5 minutes:** Analyze the degraded image
- **8 minutes:** Apply restoration techniques
- **2 minutes:** Evaluate and document results

In [ ]:
# Create the challenge image with multiple degradations
np.random.seed(42)  # For reproducible results

# Start with a high-quality image
original_challenge = data.astronaut()
gray_challenge = cv2.cvtColor(original_challenge, cv2.COLOR_RGB2GRAY)

# Step 1: Add motion blur
motion_size = 21
motion_kernel = np.zeros((motion_size, motion_size))
motion_kernel[motion_size//2, :] = 1/motion_size  # Horizontal motion
blurred_challenge = signal.convolve2d(gray_challenge, motion_kernel, mode='same', boundary='symm')

# Step 2: Add Gaussian noise
noise_level = 15
noisy_challenge = blurred_challenge + np.random.normal(0, noise_level, blurred_challenge.shape)
noisy_challenge = np.clip(noisy_challenge, 0, 255)

# Step 3: Add perspective distortion
h, w = noisy_challenge.shape
src_corners = np.array([[0, 0], [w-1, 0], [0, h-1], [w-1, h-1]], dtype=np.float32)
dst_corners = np.array([[30, 20], [w-10, 30], [20, h-40], [w-30, h-20]], dtype=np.float32)

perspective_matrix = cv2.getPerspectiveTransform(src_corners, dst_corners)
degraded_challenge = cv2.warpPerspective(noisy_challenge, perspective_matrix, (w, h))

# Display the challenge
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('🎯 RESTORATION CHALLENGE', fontsize=16, fontweight='bold', color='red')

axes[0, 0].imshow(original_challenge)
axes[0, 0].set_title('Original (Color Reference)')
axes[0, 0].axis('off')

axes[0, 1].imshow(gray_challenge, cmap='gray')
axes[0, 1].set_title('Original (Grayscale)')
axes[0, 1].axis('off')

axes[1, 0].imshow(degraded_challenge, cmap='gray')
axes[1, 0].set_title('🚨 DEGRADED IMAGE TO RESTORE 🚨', color='red', fontweight='bold')
axes[1, 0].axis('off')

# Show degradation steps
axes[1, 1].text(0.1, 0.8, '🔍 DEGRADATIONS APPLIED:', fontsize=12, fontweight='bold', transform=axes[1, 1].transAxes)
axes[1, 1].text(0.1, 0.7, '• Motion blur (horizontal)', fontsize=10, transform=axes[1, 1].transAxes)
axes[1, 1].text(0.1, 0.6, '• Gaussian noise (σ=15)', fontsize=10, transform=axes[1, 1].transAxes)
axes[1, 1].text(0.1, 0.5, '• Perspective distortion', fontsize=10, transform=axes[1, 1].transAxes)
axes[1, 1].text(0.1, 0.3, '🎯 YOUR GOALS:', fontsize=12, fontweight='bold', transform=axes[1, 1].transAxes)
axes[1, 1].text(0.1, 0.2, '• PSNR > 25 dB', fontsize=10, transform=axes[1, 1].transAxes)
axes[1, 1].text(0.1, 0.1, '• SSIM > 0.8', fontsize=10, transform=axes[1, 1].transAxes)
axes[1, 1].text(0.1, 0.0, '• Visual quality', fontsize=10, transform=axes[1, 1].transAxes)
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

# Calculate initial quality metrics
initial_psnr = calculate_psnr(gray_challenge, degraded_challenge)
initial_ssim = calculate_ssim(gray_challenge.astype(np.uint8), degraded_challenge.astype(np.uint8))

print("🚨 CHALLENGE IMAGE CREATED!")
print(f"\n📊 Initial Quality (Degraded vs Original):")
print(f"  PSNR: {initial_psnr:.2f} dB")
print(f"  SSIM: {initial_ssim:.3f}")
print(f"\n⏰ You have 15 minutes to restore this image!")
print(f"\n🎯 Target: PSNR > 25 dB, SSIM > 0.8")

# Store challenge data for student use
challenge_image = degraded_challenge.copy()
reference_image = gray_challenge.copy()

In [ ]:
# 🎯 STUDENT SOLUTION SPACE
# Your code goes here! Follow these suggested steps:

# Step 1: Analyze the degraded image
print("🔍 STEP 1: Image Analysis")
print(f"Image shape: {challenge_image.shape}")
print(f"Data type: {challenge_image.dtype}")
print(f"Value range: [{challenge_image.min():.1f}, {challenge_image.max():.1f}]")

# TODO: Add your analysis code here
# Hint: Look at the frequency domain representation
# Hint: Examine noise characteristics

# Step 2: Geometric correction (if needed)
print("\n🔄 STEP 2: Geometric Correction")
# TODO: Correct perspective distortion
# Hint: You might need to estimate or reverse the transformation
geometrically_corrected = challenge_image.copy()  # Replace with your correction

# Step 3: Deblurring
print("\n🌊 STEP 3: Deblurring")
# TODO: Apply frequency domain or iterative deblurring
# Hint: Try Wiener filter or Richardson-Lucy
deblurred = geometrically_corrected.copy()  # Replace with your deblurring

# Step 4: Noise reduction
print("\n🔧 STEP 4: Noise Reduction")
# TODO: Apply appropriate spatial filter
# Hint: Consider bilateral filter for edge preservation
final_restored = deblurred.copy()  # Replace with your noise reduction

# Step 5: Evaluation
print("\n📊 STEP 5: Quality Evaluation")
final_psnr = calculate_psnr(reference_image, final_restored)
final_ssim = calculate_ssim(reference_image.astype(np.uint8), final_restored.astype(np.uint8))

print(f"\n🎯 FINAL RESULTS:")
print(f"  PSNR: {final_psnr:.2f} dB (Target: >25 dB) {'✅' if final_psnr > 25 else '❌'}")
print(f"  SSIM: {final_ssim:.3f} (Target: >0.8) {'✅' if final_ssim > 0.8 else '❌'}")
print(f"  PSNR Improvement: {final_psnr - initial_psnr:.2f} dB")
print(f"  SSIM Improvement: {final_ssim - initial_ssim:.3f}")

# Visualize your results
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Your Restoration Results', fontsize=14, fontweight='bold')

axes[0].imshow(reference_image, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(challenge_image, cmap='gray')
axes[1].set_title(f'Degraded\nPSNR: {initial_psnr:.1f} dB')
axes[1].axis('off')

axes[2].imshow(final_restored, cmap='gray')
axes[2].set_title(f'Your Restoration\nPSNR: {final_psnr:.1f} dB')
axes[2].axis('off')

# Difference image
difference = np.abs(reference_image.astype(float) - final_restored.astype(float))
axes[3].imshow(difference, cmap='hot')
axes[3].set_title(f'Error Map\nMean Error: {np.mean(difference):.1f}')
axes[3].axis('off')

plt.tight_layout()
plt.show()

# Success evaluation
if final_psnr > 25 and final_ssim > 0.8:
    print("\n🎉 CONGRATULATIONS! You've successfully completed the challenge!")
elif final_psnr > 25 or final_ssim > 0.8:
    print("\n👍 Good progress! You've met one of the targets.")
else:
    print("\n💪 Keep working! Try different approaches or parameter tuning.")

---
## 💡 Solution & Discussion

**Click to reveal the instructor solution after attempting the challenge:**

In [ ]:
#@title 🔓 Instructor Solution (Run after your attempt)

print("🎓 INSTRUCTOR SOLUTION")
print("=" * 50)

# Step 1: Geometric correction
print("\n🔄 Step 1: Correcting perspective distortion...")
# Reverse the perspective transformation
h, w = challenge_image.shape
src_corners = np.array([[30, 20], [w-10, 30], [20, h-40], [w-30, h-20]], dtype=np.float32)
dst_corners = np.array([[0, 0], [w-1, 0], [0, h-1], [w-1, h-1]], dtype=np.float32)

inverse_perspective = cv2.getPerspectiveTransform(src_corners, dst_corners)
geom_corrected = cv2.warpPerspective(challenge_image, inverse_perspective, (w, h))

# Step 2: Motion deblurring using Richardson-Lucy
print("🌊 Step 2: Deblurring with Richardson-Lucy...")
# Recreate the motion kernel
motion_size = 21
estimated_kernel = np.zeros((motion_size, motion_size))
estimated_kernel[motion_size//2, :] = 1/motion_size

# Apply Richardson-Lucy deconvolution
deblurred_solution = richardson_lucy(np.maximum(geom_corrected, 0), estimated_kernel, iterations=15)

# Step 3: Noise reduction with bilateral filter
print("🔧 Step 3: Noise reduction with bilateral filtering...")
# Convert to uint8 for bilateral filter
deblurred_uint8 = np.clip(deblurred_solution, 0, 255).astype(np.uint8)
denoised_solution = cv2.bilateralFilter(deblurred_uint8, 9, 75, 75)

# Step 4: Final enhancement with unsharp masking
print("✨ Step 4: Final enhancement...")
gaussian_smooth = ndimage.gaussian_filter(denoised_solution, sigma=1.0)
unsharp_mask = denoised_solution.astype(float) - gaussian_smooth
final_solution = denoised_solution.astype(float) + 0.5 * unsharp_mask
final_solution = np.clip(final_solution, 0, 255)

# Evaluate solution
solution_psnr = calculate_psnr(reference_image, final_solution)
solution_ssim = calculate_ssim(reference_image.astype(np.uint8), final_solution.astype(np.uint8))

print(f"\n📊 INSTRUCTOR SOLUTION RESULTS:")
print(f"  PSNR: {solution_psnr:.2f} dB")
print(f"  SSIM: {solution_ssim:.3f}")
print(f"  Improvement from degraded: {solution_psnr - initial_psnr:.2f} dB PSNR")

# Visualize step-by-step restoration
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Instructor Solution: Step-by-Step Restoration', fontsize=14, fontweight='bold')

steps = [
    ('Original', reference_image),
    ('Degraded', challenge_image),
    ('Geom. Corrected', geom_corrected),
    ('Deblurred', deblurred_solution),
    ('Denoised', denoised_solution),
    ('Final Result', final_solution)
]

for i, (title, img) in enumerate(steps):
    row, col = i // 3, i % 3
    axes[row, col].imshow(img, cmap='gray')
    
    if title not in ['Original', 'Degraded']:
        psnr_step = calculate_psnr(reference_image, img)
        axes[row, col].set_title(f'{title}\nPSNR: {psnr_step:.1f} dB')
    else:
        axes[row, col].set_title(title)
    
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("\n🎓 KEY INSIGHTS FROM SOLUTION:")
print("  • Order matters: Geometric correction → Deblurring → Denoising")
print("  • Richardson-Lucy preserves edges better than Wiener filter")
print("  • Bilateral filter removes noise while preserving important features")
print("  • Unsharp masking provides final sharpening enhancement")
print("  • Parameter tuning is crucial for optimal results")

## 🎯 Workshop Summary

Congratulations! You've completed the Image Restoration and Geometric Processing workshop. Let's summarize what we've learned:

### 🔑 Key Concepts Covered:

1. **Image Degradation Model**: $g(x,y) = h(x,y) * f(x,y) + \eta(x,y)$
2. **Spatial Domain Restoration**: Mean, Gaussian, Median, and Bilateral filtering
3. **Frequency Domain Restoration**: Wiener filtering and Richardson-Lucy deconvolution
4. **Geometric Processing**: Transformations, interpolation, and registration
5. **Quality Assessment**: PSNR and SSIM metrics

### 🎓 Learning Outcomes Achieved:

✅ **Understanding**: Degradation models and restoration principles  
✅ **Application**: Various restoration techniques in practice  
✅ **Implementation**: Hands-on coding of restoration algorithms  
✅ **Performance**: Geometric transformations and image alignment  
✅ **Evaluation**: Quantitative quality assessment methods  
✅ **Problem-solving**: Real-world restoration challenges  

### 🔬 Best Practices Learned:

- **Order of operations matters** in multi-step restoration
- **Parameter tuning** is crucial for optimal results
- **Method selection** depends on degradation type
- **Quality metrics** guide restoration effectiveness
- **Edge preservation** is important in practical applications

### 🚀 Next Steps:

1. **Explore advanced methods**: Deep learning-based restoration
2. **Study specialized applications**: Medical imaging, satellite imagery
3. **Practice on real data**: Download and restore actual degraded images
4. **Implement optimizations**: Real-time processing techniques
5. **Research current topics**: Computational photography, HDR imaging

---

### 📚 Additional Resources:

- **Course Materials**: Check the slides and figure demonstrations
- **Documentation**: scikit-image, OpenCV restoration modules
- **Papers**: Classic papers on Wiener filtering and Richardson-Lucy
- **Datasets**: Berkeley Segmentation Dataset, DIV2K for practice

---

**Great work! You're now equipped with fundamental and advanced image restoration techniques! 🎉**

---

## 📝 Workshop Credits

**CMSC 178IP - Digital Image Processing**  
**Topic:** Image Restoration and Geometric Processing  
**Author:** Noel Jeffrey Pinton  
**Institution:** University of the Philippines - Cebu  
**Department:** Computer Science  

### 🙏 Acknowledgments:
- scikit-image and OpenCV communities for excellent documentation
- Classic papers by Wiener, Richardson, and Lucy
- Students who helped refine this workshop content

### 📧 Contact & Feedback:
- Issues and improvements: [GitHub Repository](https://github.com/njpinton/CMSC178IP)
- Course questions: Through your learning management system

---

*Last updated: 2024 | Version 1.0*